# Clustering UMAP + HDBSCAN

Notebook per usare le funzioni definite in `src/clustering/hdbscan.py` e `src/visualization/plot.py`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.clustering.hdbscan import evaluate_clustered_data, optimize_umap_hdbscan_parameters
from src.visualization.plot import plot_clustering_on_umap
from src.visualization.sample import show_clustering_samples

pd.set_option("display.max_columns", None)

## Caricamento embedding

Scegli il file `.npy` da clusterizzare modificando `EMBEDDING_PATH`.

In [ ]:
EMBEDDINGS_DIR = PROJECT_ROOT / "data" / "glomeruli" / "embeddings"
embedding_files = sorted(EMBEDDINGS_DIR.glob("*.npy"))

if not embedding_files:
    raise FileNotFoundError(f"Nessun file .npy trovato in {EMBEDDINGS_DIR}")

EMBEDDING_PATH = EMBEDDINGS_DIR / "dinov3_small_plus_patch_masked_embeddings.npy"
embeddings = np.load(EMBEDDING_PATH)

print("\nFile selezionato:", EMBEDDING_PATH.name)
print("Shape embeddings:", embeddings.shape)

In [ ]:
metadata_path = EMBEDDING_PATH.with_suffix(".csv")

if metadata_path.exists():
    metadata = pd.read_csv(metadata_path).sort_values("index")
    image_paths = [
        Path(path) if Path(path).is_absolute() else PROJECT_ROOT / path
        for path in metadata["image_path"]
    ]
    print(f"Metadata caricati: {len(image_paths)} immagini")
else:
    metadata = None
    image_paths = []
    raise FileNotFoundError("Metadata CSV non trovato: continuo solo con embedding e label.")

## Preprocessing

HDBSCAN lavora meglio dopo standardizzazione e spesso dopo PCA. Qui `X` diventa l'input passato alla funzione di ottimizzazione in `hdbscan.py`.

In [ ]:
USE_PCA = True
PCA_VARIANCE = 0.95
RANDOM_STATE = 42

X_scaled = StandardScaler().fit_transform(embeddings)

if USE_PCA:
    pca = PCA(n_components=PCA_VARIANCE, random_state=RANDOM_STATE)
    X = pca.fit_transform(X_scaled)
    print("Shape dopo PCA:", X.shape)
    print("Varianza spiegata:", pca.explained_variance_ratio_.sum())
else:
    X = X_scaled
    print("Shape senza PCA:", X.shape)

## Ottimizzazione UMAP + HDBSCAN

Questa cella usa `optimize_umap_hdbscan_parameters` da `src/clustering/hdbscan.py`. La griglia sotto e' volutamente piccola; puoi ampliarla quando vuoi fare una ricerca piu' completa.

In [ ]:
optimization = optimize_umap_hdbscan_parameters(
    X,
    umap_n_neighbors_values=(15, 20, 25, 30, 40),
    umap_min_dist_values=(0.1, 0.05, 0.1),
    umap_n_components_values=(5, 10, 15, 20, 25, 30, 35, 40),
    umap_metric_values=("euclidean",),
    hdbscan_min_cluster_sizes = (10, 15, 20, 30),
    hdbscan_min_samples_values = (3, 5, 10, 15),
    hdbscan_cluster_selection_methods=("eom",),
    min_clusters=8,
    max_clusters=13,
    max_noise_ratio=0.33,
    random_state=RANDOM_STATE,
)

results_df = optimization["results"]
display(results_df.head(10))

In [ ]:
best_labels = optimization["best_labels"]
best_umap_embedding = optimization["best_embedding"]
best_params = optimization["best_params"]

if best_labels is None:
    raise RuntimeError("Nessuna configurazione valida trovata. Amplia la griglia o aumenta max_noise_ratio.")

best_metrics = evaluate_clustered_data(
    X=best_umap_embedding,
    labels=best_labels,
    noise_label=-1,
    ignore_noise=True,
    silhouette_metric="euclidean",
)

print("Best params:")
display(best_params)

print("Best metrics:")
display(pd.Series(best_metrics))

cluster_counts = pd.Series(best_labels).value_counts().sort_index()
display(cluster_counts.rename("n_samples").to_frame())

## Plot dei cluster su UMAP

Queste celle usano `plot_clustering_on_umap` da `src/visualization/plot.py`. La funzione ricalcola UMAP internamente in base a `mode`: `2d` oppure `3d`.

In [ ]:
plot_clustering_on_umap(
    X,
    best_labels,
    mode="2d",
    title=f"{EMBEDDING_PATH.stem} - HDBSCAN su UMAP 2D",
    n_neighbors=best_params["umap"]["n_neighbors"],
    min_dist=best_params["umap"]["min_dist"],
    metric=best_params["umap"]["metric"],
    random_state=RANDOM_STATE,
);

In [ ]:
plot_clustering_on_umap(
    X,
    best_labels,
    mode="3d",
    title=f"{EMBEDDING_PATH.stem} - HDBSCAN su UMAP 3D",
    n_neighbors=best_params["umap"]["n_neighbors"],
    min_dist=best_params["umap"]["min_dist"],
    metric=best_params["umap"]["metric"],
    random_state=RANDOM_STATE,
);

## Sample per cluster

Questa sezione usa `show_clustering_samples` da `src/visualization/sample.py`. HDBSCAN viene rifittato sul miglior embedding UMAP per ottenere `probabilities_`, poi le immagini vengono ordinate per probabilita' dentro ogni cluster.

In [ ]:
sample_clusterer = HDBSCAN(
    min_cluster_size=best_params["hdbscan"]["min_cluster_size"],
    min_samples=best_params["hdbscan"]["min_samples"],
    metric=best_params["hdbscan"]["metric"],
    cluster_selection_method=best_params["hdbscan"]["cluster_selection_method"],
    allow_single_cluster=False,
    n_jobs=-1,
)

sample_labels = sample_clusterer.fit_predict(best_umap_embedding)
sample_probabilities = sample_clusterer.probabilities_

if not np.array_equal(sample_labels, best_labels):
    print("Nota: le label rifittate per ottenere probabilities_ differiscono da best_labels; uso sample_labels per i sample.")

if not image_paths:
    raise RuntimeError("image_paths non disponibili: serve il CSV metadata per mostrare i sample.")

sample_figure, sample_axes = show_clustering_samples(
    labels=sample_labels,
    probabilities=sample_probabilities,
    image_paths=image_paths,
    x=20,
    include_noise=False,
    title=f"{EMBEDDING_PATH.stem} - prime immagini per cluster",
)
sample_figure.savefig(PROJECT_ROOT / "dinov3_small_plus_patch_masked.png")
sample_figure;